In [0]:
%sql
-- Set the catalog first
USE CATALOG workspace;
-- Then use the schema
USE SCHEMA bronze;

In [0]:
base_path = "/Volumes/workspace/bronze/ecom_datalakehouse"



In [0]:
from pyspark.sql.functions import current_timestamp, lit

volume_path = "/Volumes/workspace/bronze/ecom_datalakehouse"

files = {
    "customers.csv": "customers",
    "orders.csv": "orders",
    "order_items.csv": "order_items",
    "products.csv": "products",
    "order_payments.csv": "payments",
    "order_reviews.csv": "reviews",
    "sellers.csv": "sellers",
    "geolocation.csv": "geolocation",
    "category_translation.csv": "category_translation"
}

for file, table in files.items():
    
    df = (
        spark.read
        .format("csv")
        .option("header", True)
        .option("inferSchema", True)
        .option("multiLine", True)  
        .option("quote", "\"")     
        .option("escape", "\"")    
        .load(f"{volume_path}/{file}")
    )

    df = (
        df
        .withColumn("bronze_ingestion_time", current_timestamp())
        .withColumn("source_file", lit(file))
    )

    (
        df.write
        .format("delta")
        .mode("overwrite")
        .option("overwriteSchema", "true")
        .saveAsTable(f"workspace.bronze.{table}")
    )

    print(f"Loaded: {table}")

In [0]:
%sql
SHOW TABLES IN workspace.bronze;




In [0]:
%sql
OPTIMIZE workspace.bronze.orders;
OPTIMIZE workspace.bronze.customers;
OPTIMIZE workspace.bronze.order_items;
OPTIMIZE workspace.bronze.products;
OPTIMIZE workspace.bronze.payments;
OPTIMIZE workspace.bronze.reviews;
OPTIMIZE workspace.bronze.sellers;
OPTIMIZE workspace.bronze.geolocation;
OPTIMIZE workspace.bronze.category_translation;

In [0]:
tables = [
    "customers",
    "orders",
    "order_items",
    "products",
    "payments",
    "reviews",
    "sellers"
]

for table in tables:
    df = spark.table(f"workspace.bronze.{table}")
    print(f"{table} row count:", df.count())